In [1]:
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import wandb

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
I0000 00:00:1776095241.781928  858540 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
wandb.init(
    project="Lab1",
    name="transf-bert",
    config={
        "learning_rate": 2e-5,
        "batch_size": 8,
        "epochs": 3,
        "model": "bert-base-cased"
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
#charge the dataset
dataset = load_dataset(
    "csv",
    data_files="amazon_cells_labelled_LARGE_25K.txt",
    delimiter="\t",
    column_names=["text", "label"])

# convert labels
dataset = dataset.map(lambda x: {"label": int(x["label"])})

#split
dataset = dataset["train"].train_test_split(test_size=0.3)

# split temp en validation + test
temp_split = dataset["test"].train_test_split(test_size=0.5)

# Tokenizer
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

dataset = {
    "train": dataset["train"],
    "validation": temp_split["train"],
    "test": temp_split["test"]
}

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_datasets = {
    split: dataset[split].map(tokenize_function, batched=True)
    for split in dataset
}
data_collator = DataCollatorWithPadding(tokenizer)

Map: 100%|██████████| 3750/3750 [00:00<00:00, 17155.22 examples/s]


In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [8]:
training_args = TrainingArguments(
    output_dir="test-trainer",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,

    learning_rate=2e-5,
    weight_decay=0.01,
    
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    report_to="wandb"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],  
    compute_metrics=compute_metrics, data_collator=data_collator                 
)
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss


In [1]:
predictions = trainer.predict(tokenized_datasets["test"])

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

cm = confusion_matrix(labels, preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.show()

NameError: name 'trainer' is not defined

In [9]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

RuntimeError: on_train_begin must be called before on_evaluate

In [10]:
wandb.finish()

eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████
+3,...
